# Composition bias: methods and how the data fits

Exploratory notebook. Reads the per-sample `.h5ad` files only; nothing is filtered, normalized in place, or saved.

**What composition bias is.** Total-count normalization assumes every cell's counts scale together. If a few genes take a large share of a
cell's counts (MZB1 in a plasma cell, SLC4A1 in an erythroid cell), the remaining genes look lower in that cell after dividing by the total,
even when their absolute expression is the same.

The question for each method: how much of the counts do a few genes take, and does that vary enough between cells to distort normalization?

| # | Method | Question it answers |
|---|---|---|
| 1 | Pooled share and cumulative curve | How top-heavy is each sample overall? |
| 2 | The median-fraction top-10 | Why the old top-genes cell can miss dominant genes |
| 3 | Per-gene dominance across cells | Which genes dominate individual cells, and how many cells? |
| 4 | Normalization sensitivity | How much would total-count normalization distort the other genes? |
| 5 | Cross-sample composition | Do the samples differ in composition enough to matter for comparing them? |
| 6 | Clustering robustness | Do the clusters, especially the plasma-cell sub-clusters, survive size factors that ignore the dominant genes? |
| 7 | Pearson residuals as a third arm | Do clusters change when the pipeline uses analytic Pearson residuals instead of log-normalization? |
| 8 | Spike-in test | Does each normalization leave the other genes alone when a dominant gene is added to a cell? |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import harmonypy as hm
import scipy.sparse as sp
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score

In [ ]:
SAMPLES_DIR = '/home/workspace/PCD_data/xenium/commercial/data/samples'

SAMPLE_NAMES = [
    'TIS05393',
    'TIS04788',
    'TIS05392',
    'TIS05394',
    'TIS04789',
    'TIS05397_008',
]

MIN_COUNTS = 15
TOP_N      = 10

## Load

Cells below `MIN_COUNTS` are dropped so per-cell fractions are not computed from a handful of counts.
`counts[name]` is the cells × genes count matrix, `fractions[name]` is the same matrix with each row divided by its row sum.

In [ ]:
samples   = {}
counts    = {}
fractions = {}
spatial   = {}
for name in SAMPLE_NAMES:
    adata = ad.read_h5ad(f'{SAMPLES_DIR}/{name}.h5ad')
    # Drops cells with fewer than MIN_COUNTS total counts; in memory only, the file on disk is not changed.
    adata = adata[np.asarray(adata.X.sum(axis=1)).ravel() >= MIN_COUNTS].copy()

    samples[name] = adata
    counts[name]  = sp.csr_matrix(adata.X)
    row_totals    = np.asarray(counts[name].sum(axis=1)).ravel()
    fractions[name] = sp.diags(1 / row_totals) @ counts[name]
    spatial[name] = adata.obsm['spatial']

GENES = samples[SAMPLE_NAMES[0]].var_names

pd.Series({name: adata.n_obs for name, adata in samples.items()}, name='cells kept')

## 1. Pooled share and cumulative curve

**How it works.** Add up each gene's counts over all cells, divide by the sample total, and rank. The cumulative share of the top k genes says how
top-heavy the sample is. The Gini coefficient summarises the whole curve: 0 means every gene has the same share, values near 1 mean a few genes take everything.

**What it can't tell.** It is a pooled number, so it is dominated by the most numerous cell type, and it says nothing about whether the share varies between cells.
A 377-gene marker panel will always look top-heavy to some degree.

In [ ]:
pooled_share = {}
share_rows   = []
for name in SAMPLE_NAMES:
    gene_totals = np.asarray(counts[name].sum(axis=0)).ravel()
    share = pd.Series(gene_totals / gene_totals.sum(), index=GENES).sort_values(ascending=False)
    pooled_share[name] = share

    ascending = np.sort(share.to_numpy())
    ranks     = np.arange(1, len(ascending) + 1)
    gini      = 2 * np.sum(ranks * ascending) / (len(ascending) * ascending.sum()) - (len(ascending) + 1) / len(ascending)

    share_rows.append({
        'sample': name,
        'top1': share.iloc[:1].sum(),
        'top5': share.iloc[:5].sum(),
        'top10': share.iloc[:10].sum(),
        'top20': share.iloc[:20].sum(),
        'gini': gini,
        'top gene': share.index[0],
    })

pd.DataFrame(share_rows).set_index('sample').round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in SAMPLE_NAMES:
    cumulative = pooled_share[name].cumsum().to_numpy()
    axes[0].plot(np.arange(1, len(cumulative) + 1), cumulative, label=name)
axes[0].plot([1, len(GENES)], [1 / len(GENES), 1], color='gray', ls=':', label='all genes equal')
axes[0].set_xscale('log')
axes[0].set_xlabel('gene rank')
axes[0].set_ylabel('cumulative share of counts')
axes[0].set_title('Cumulative share by gene rank')
axes[0].legend(fontsize=8)

for name in SAMPLE_NAMES:
    axes[1].plot(np.arange(1, 31), pooled_share[name].iloc[:30].to_numpy(), marker='o', ms=3, label=name)
axes[1].set_yscale('log')
axes[1].set_xlabel('gene rank')
axes[1].set_ylabel('share of counts')
axes[1].set_title('Top 30 genes')

plt.show()

## 2. The median-fraction top-10

**How it works.** The existing top-genes cell takes each gene's median per-cell fraction and plots the ten largest.

**What it can't tell.** If a gene is zero in more than half of the cells, its median is 0 however dominant it is in the cells that do express it.
When almost every gene has a median of 0, the ranking is a tie and the "top 10" is decided by gene order, not by expression.
The table shows how many genes have a non-zero median in each sample, and how the median-based top 10 overlaps the pooled top 10.

In [ ]:
median_rows = []
for name in SAMPLE_NAMES:
    median_fraction = np.median(fractions[name].toarray(), axis=0)
    top_by_median   = GENES[np.argsort(median_fraction)[-TOP_N:][::-1]]
    top_by_pooled   = pooled_share[name].index[:TOP_N]
    median_rows.append({
        'sample': name,
        'genes with median > 0': int((median_fraction > 0).sum()),
        'overlap with pooled top 10': len(set(top_by_median) & set(top_by_pooled)),
        'top 3 by median': ', '.join(top_by_median[:3]),
        'top 3 pooled': ', '.join(top_by_pooled[:3]),
    })

pd.DataFrame(median_rows).set_index('sample')

## 3. Per-gene dominance across cells

**How it works.** For each gene, count the fraction of cells where that gene alone takes more than a threshold share (5%, 10%, 25%) of the cell's counts.
This finds genes that dominate a subset of cells, which the pooled and median views both hide.

**What it can't tell.** The threshold is a choice, and a high value can be real biology (a plasma cell really does make mostly immunoglobulin-pathway transcripts).
It identifies where composition varies, not whether that is a problem. Section 4 asks that.

Cells here carry only a few tens of counts (median about 15 to 30), so per-cell fractions are coarse: in a cell with 15 counts, one transcript is 6.7% and two are 13%. Low thresholds mostly pick up single transcripts; the 25% threshold and the per-cell curve below are more meaningful. Raising `MIN_COUNTS` makes fractions less noisy at the cost of dropping cells.

In [ ]:
THRESHOLDS = [0.05, 0.10, 0.25]

dominance = {}
for name in SAMPLE_NAMES:
    table = pd.DataFrame(index=GENES)
    for threshold in THRESHOLDS:
        table[f'>{threshold:.0%}'] = np.asarray((fractions[name] > threshold).sum(axis=0)).ravel() / samples[name].n_obs
    dominance[name] = table

top_dominant_genes = []
for name in SAMPLE_NAMES:
    top_dominant_genes.extend(dominance[name]['>10%'].sort_values(ascending=False).index[:8])
top_dominant_genes = list(dict.fromkeys(top_dominant_genes))

heatmap = pd.DataFrame({name: dominance[name].loc[top_dominant_genes, '>10%'] for name in SAMPLE_NAMES})

fig, ax = plt.subplots(figsize=(7, 0.35 * len(heatmap) + 1.5))
image = ax.imshow(heatmap.to_numpy(), cmap='viridis', aspect='auto')
ax.set_xticks(range(len(SAMPLE_NAMES)), SAMPLE_NAMES, rotation=45, ha='right')
ax.set_yticks(range(len(heatmap)), heatmap.index)
for row in range(heatmap.shape[0]):
    for column in range(heatmap.shape[1]):
        ax.text(column, row, f'{heatmap.iloc[row, column]:.2f}', ha='center', va='center', color='white', fontsize=7)
fig.colorbar(image, ax=ax, label='fraction of cells where gene takes >10% of counts')
ax.set_title('Per-gene dominance')
plt.show()

**Per cell.** The largest single-gene fraction in each cell, and which gene it is. The curve shows how many cells have one gene taking more than a given share.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
dominant_rows = []
for name in SAMPLE_NAMES:
    largest_fraction = np.asarray(fractions[name].max(axis=1).todense()).ravel()
    dominant_gene    = GENES[np.asarray(fractions[name].argmax(axis=1)).ravel()]

    sorted_fraction = np.sort(largest_fraction)
    ax.plot(sorted_fraction, 1 - np.arange(1, len(sorted_fraction) + 1) / len(sorted_fraction), label=name)

    over_quarter = pd.Series(dominant_gene[largest_fraction > 0.25]).value_counts()
    dominant_rows.append({
        'sample': name,
        'median largest fraction': np.median(largest_fraction),
        'cells with one gene > 25%': (largest_fraction > 0.25).mean(),
        'most common dominant genes': ', '.join(f'{gene} ({count})' for gene, count in over_quarter.iloc[:4].items()),
    })

ax.set_xlabel('largest single-gene fraction in a cell')
ax.set_ylabel('fraction of cells above')
ax.set_title('Per-cell dominance')
ax.legend(fontsize=8)
plt.show()

pd.DataFrame(dominant_rows).set_index('sample').round(3)

## 4. Normalization sensitivity

**How it works.** For each sample, take the top `TOP_N` genes by pooled share as the dominant set. For every cell, `top_fraction` is the share of its counts that
come from those genes. Normalizing by total counts versus normalizing by the counts outside the dominant set rescales every other gene in a cell by
`1 / (1 - top_fraction)`. Cells with different `top_fraction` therefore get different scaling of the same genes. The distortion fold is that factor relative to the sample's median cell.

The scanpy option `exclude_highly_expressed=True` is the usual fix, but it drops any gene that exceeds `max_fraction` in *any* cell.
The last column shows how many of the genes that rule would drop here. It reports every gene, because with a median of tens of counts per cell a single transcript already exceeds 5% of some cell, so the rule cannot be used on this data as it stands.

**What it can't tell.** It measures how large the effect is, not whether the dominant genes should be excluded.
`top_fraction` is mostly set by cell type, so large spread is expected in mixed tissue; whether it needs correcting depends on which comparisons you make downstream.

In [ ]:
MAX_FRACTION = 0.05

top_fraction = {}
sensitivity_rows = []
for name in SAMPLE_NAMES:
    top_genes   = pooled_share[name].index[:TOP_N]
    top_columns = np.where(GENES.isin(top_genes))[0]
    top_fraction[name] = np.asarray(fractions[name][:, top_columns].sum(axis=1)).ravel()

    remaining = 1 - top_fraction[name]
    with np.errstate(divide='ignore'):
        fold = remaining / np.median(remaining)
        distortion = np.maximum(fold, 1 / fold)

    total_counts = np.asarray(counts[name].sum(axis=1)).ravel()
    sensitivity_rows.append({
        'sample': name,
        'median top_fraction': np.median(top_fraction[name]),
        'p90 top_fraction': np.quantile(top_fraction[name], 0.9),
        'cells > 1.5x distortion': (distortion > 1.5).mean(),
        'cells > 2x distortion': (distortion > 2).mean(),
        'spearman with total counts': spearmanr(top_fraction[name], total_counts)[0],
        f'genes dropped by scanpy rule ({MAX_FRACTION:.0%})': int((fractions[name].max(axis=0).toarray().ravel() > MAX_FRACTION).sum()),
    })

pd.DataFrame(sensitivity_rows).set_index('sample').round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, name in zip(axes.ravel(), SAMPLE_NAMES):
    ax.hist(top_fraction[name], bins=60, color='steelblue')
    ax.set_xlabel(f'fraction of counts from top {TOP_N} genes')
    ax.set_title(f'{name} — median {np.median(top_fraction[name]):.2f}')

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, name in zip(axes.ravel(), SAMPLE_NAMES):
    coordinates = spatial[name]
    bins = ax.hexbin(
        coordinates[:, 0], coordinates[:, 1], C=top_fraction[name],
        reduce_C_function=np.mean, gridsize=120, cmap='viridis', vmin=0, vmax=1, rasterized=True,
    )
    fig.colorbar(bins, ax=ax, shrink=0.6)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_axis_off()
    ax.set_title(f'{name} — mean top-{TOP_N} fraction')

fig.tight_layout()
plt.show()

## 5. Cross-sample composition

**How it works.** Sum each sample into a pseudobulk profile and compute DESeq2-style median-of-ratios size factors: each gene's count in a sample divided by its geometric
mean across samples, then the median of those ratios. This ignores genes that behave differently, so it is robust to a few dominant genes.
Compare that with plain library size. If they agree, the samples differ only in depth. If they diverge, composition differs and library-size scaling would misrepresent the other genes.

**What it can't tell.** Only six samples, from different tissues, so differences are largely real biology (plasma-cell rich versus erythroid rich), not an artifact.
This is a check on whether comparing samples after library-size normalization is safe.

In [ ]:
pseudobulk = pd.DataFrame({name: np.asarray(counts[name].sum(axis=0)).ravel() for name in SAMPLE_NAMES}, index=GENES)
pseudobulk = pseudobulk[(pseudobulk > 0).all(axis=1)]

log_geometric_mean = np.log(pseudobulk).mean(axis=1)
ratio_size_factor  = np.exp(np.median(np.log(pseudobulk).sub(log_geometric_mean, axis=0), axis=0))
library_size       = pseudobulk.sum(axis=0).to_numpy()

comparison = pd.DataFrame({
    'library size': library_size,
    'library size, relative': library_size / np.exp(np.log(library_size).mean()),
    'median-of-ratios, relative': ratio_size_factor / np.exp(np.log(ratio_size_factor).mean()),
}, index=SAMPLE_NAMES)
comparison['composition effect'] = comparison['median-of-ratios, relative'] / comparison['library size, relative']

comparison.round(3)

## 6. Clustering robustness

**The question.** Your processing divides every cell by its total counts. In cells where a few genes take a large share, that rescales all other genes. If the sub-clusters among the neoplastic plasma cells
are partly a dominance gradient and not biology, they should change when the size factors ignore the dominant genes.

**How it works.** `combined_clustered.h5ad` is loaded read-only and its raw `layers['counts']` are run through the same steps as `03_processing`
(normalize, log1p, scale to unit variance clipped at 10, PCA with 30 components, Harmony by sample, 15 neighbours, Leiden at resolution 1.1) twice, with identical settings and seeds:

- **total counts** is the control. It is your current normalization. Comparing it with your saved clusters shows how closely this rerun reproduces them, so any drop below that is not a normalization effect.
- **excluding dominant genes** uses size factors from counts outside the top `TOP_N` genes. The dominant genes stay in the feature matrix; only the scaling changes.

Both are compared with your saved `leiden_res1.10` clusters and `cell_type_res1_1` names using the adjusted Rand index (1 is identical, 0 is chance), overall and within the neoplastic plasma cells.

**What it can't tell.** Leiden is sensitive to small changes, so some disagreement is expected even for the control. Cells with almost no counts outside the dominant genes get a very large scaling factor,
and cells with none are dropped (counted in the table). The dominant set is the global top `TOP_N` genes, not per sample. Nothing is saved: all objects stay in memory.

In [ ]:
CLUSTERED_PATH = '/home/workspace/PCD_data/xenium/commercial/data/objects/combined_clustered.h5ad'
RESOLUTION     = 1.1
N_PCS          = 30

clustered  = ad.read_h5ad(CLUSTERED_PATH)
raw_counts = sp.csr_matrix(clustered.layers['counts'])

original_clusters = clustered.obs['leiden_res1.10'].astype(str).to_numpy()
cell_types        = clustered.obs['cell_type_res1_1'].astype(str).to_numpy()
samples_per_cell  = clustered.obs[['sample']].copy()
is_plasma         = clustered.obs['cell_type_res1_1'].astype(str).str.startswith('Neoplastic').to_numpy()

gene_totals      = np.asarray(raw_counts.sum(axis=0)).ravel()
dominant_columns = np.argsort(gene_totals)[::-1][:TOP_N]
outside_columns  = np.setdiff1d(np.arange(raw_counts.shape[1]), dominant_columns)

size_factors = {
    'total counts': np.asarray(raw_counts.sum(axis=1)).ravel(),
    'excluding dominant genes': np.asarray(raw_counts[:, outside_columns].sum(axis=1)).ravel(),
}
top_fraction_all = np.asarray(raw_counts[:, dominant_columns].sum(axis=1)).ravel() / size_factors['total counts']

pd.DataFrame({
    'dominant genes': ', '.join(clustered.var_names[dominant_columns]),
    'cells': clustered.n_obs,
    'neoplastic plasma cells': int(is_plasma.sum()),
}, index=['clustered object'])

**Rerun both pipelines.** This is the slow cell: each rerun runs PCA, Harmony, neighbours and Leiden on all cells.

In [ ]:
reruns     = {}
keep_masks = {}
for label, size_factor in size_factors.items():
    keep = size_factor > 0
    target_sum = np.median(size_factor[keep])
    # Builds a new in-memory AnnData from the raw counts, each cell rescaled to target_sum using this size factor;
    # the clustered object and its files are not changed.
    rescaled = (sp.diags(target_sum / size_factor[keep]) @ raw_counts[keep]).tocsr().astype(np.float32)
    rerun = ad.AnnData(X=rescaled, obs=samples_per_cell[keep].copy(), var=pd.DataFrame(index=clustered.var_names))

    # Same steps and settings as 03_processing, applied to this in-memory copy only.
    sc.pp.log1p(rerun)
    sc.pp.scale(rerun, zero_center=False, max_value=10)
    sc.pp.pca(rerun, n_comps=N_PCS, use_highly_variable=False, random_state=0)

    harmony = hm.run_harmony(rerun.obsm['X_pca'].astype('float64'), rerun.obs, 'sample', random_state=0)
    harmony_pca = harmony.Z_corr
    if harmony_pca.shape[0] != rerun.n_obs:
        harmony_pca = harmony_pca.T
    rerun.obsm['X_pca_harmony'] = harmony_pca

    sc.pp.neighbors(rerun, use_rep='X_pca_harmony', n_neighbors=15, n_pcs=N_PCS, random_state=0)
    sc.tl.leiden(rerun, flavor='igraph', n_iterations=2, directed=False,
                 resolution=RESOLUTION, key_added='leiden_rerun', random_state=0)

    reruns[label]     = rerun
    keep_masks[label] = keep

**Agreement with your saved clusters.** The control row is the ceiling: how well this rerun reproduces your clusters with no change in normalization.
The last column is the strongest Spearman correlation between the dominant-gene fraction and the first five PCs.

In [ ]:
summary_rows = []
for label, rerun in reruns.items():
    keep  = keep_masks[label]
    new   = rerun.obs['leiden_rerun'].astype(str).to_numpy()
    plasma_kept = is_plasma[keep]
    pcs = rerun.obsm['X_pca']
    summary_rows.append({
        'size factors': label,
        'cells dropped': int((~keep).sum()),
        'clusters': len(np.unique(new)),
        'ARI vs saved clusters': adjusted_rand_score(original_clusters[keep], new),
        'ARI vs saved cell types': adjusted_rand_score(cell_types[keep], new),
        'ARI, plasma cells only': adjusted_rand_score(cell_types[keep][plasma_kept], new[plasma_kept]),
        'max |spearman| top fraction vs PC1-5': max(abs(spearmanr(top_fraction_all[keep], pcs[:, i])[0]) for i in range(5)),
    })

pd.DataFrame(summary_rows).set_index('size factors').round(3)

**Where the plasma-cell sub-clusters go.** Each row is one of your saved neoplastic plasma-cell types; each column is a cluster from the rerun (only clusters holding more than 5% of some type).
A row concentrated in one column means that type survives. A row spread over several, or two rows sharing one column, means the split moved.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
best_match = {}
for ax, (label, rerun) in zip(axes, reruns.items()):
    keep        = keep_masks[label]
    plasma_kept = is_plasma[keep]
    new         = rerun.obs['leiden_rerun'].astype(str).to_numpy()

    table = pd.crosstab(cell_types[keep][plasma_kept], new[plasma_kept], normalize='index')
    best_match[label] = table.max(axis=1)
    table = table.loc[:, table.max(axis=0) > 0.05]

    image = ax.imshow(table.to_numpy(), cmap='viridis', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(table.shape[1]), table.columns)
    ax.set_yticks(range(table.shape[0]), [name[:38] for name in table.index] if ax is axes[0] else [])
    for row in range(table.shape[0]):
        for column in range(table.shape[1]):
            ax.text(column, row, f'{table.iloc[row, column]:.2f}', ha='center', va='center', color='white', fontsize=7)
    ax.set_xlabel('cluster in rerun')
    ax.set_title(f'Size factors: {label}')

fig.colorbar(image, ax=axes, label='fraction of the saved type', shrink=0.8)
plt.show()

pd.DataFrame(best_match).round(2)

**The two reruns against each other.** If the control does not reproduce your saved clusters (low ARI above), the saved clusters are the wrong reference, for example because the saved object came
from an earlier version of the processing. Here the control rerun is the reference instead. The noise floor reclusters the control's own graph with a different Leiden seed, so it shows how much
agreement to expect when nothing about the normalization changes.

In [ ]:
control = reruns['total counts']
excluding = reruns['excluding dominant genes']
control_clusters   = control.obs['leiden_rerun'].astype(str).to_numpy()
excluding_clusters = excluding.obs['leiden_rerun'].astype(str).to_numpy()

# Adds obs['leiden_seed1'] to the in-memory control rerun: the same graph clustered with Leiden seed 1; nothing is saved.
sc.tl.leiden(control, flavor='igraph', n_iterations=2, directed=False,
             resolution=RESOLUTION, key_added='leiden_seed1', random_state=1)
control_seed1_clusters = control.obs['leiden_seed1'].astype(str).to_numpy()

pd.DataFrame({
    'all cells': {
        'control vs control, other Leiden seed (noise floor)': adjusted_rand_score(control_clusters, control_seed1_clusters),
        'control vs excluding dominant genes': adjusted_rand_score(control_clusters, excluding_clusters),
    },
    'plasma cells only': {
        'control vs control, other Leiden seed (noise floor)': adjusted_rand_score(control_clusters[is_plasma], control_seed1_clusters[is_plasma]),
        'control vs excluding dominant genes': adjusted_rand_score(control_clusters[is_plasma], excluding_clusters[is_plasma]),
    },
}).round(3)

**Reading the result.**

- The comparison that matters is *control vs excluding dominant genes* against the noise floor above it. If the two are close, changing the size factors moves the clusters no more than re-running Leiden with another seed does, and the sub-clusters do not depend on the scaling.
  If the first is clearly lower, the scaling changes the clusters, and the heatmap shows which plasma-cell types move.
- The first summary table compares each rerun with your saved clusters. A low control row there means the rerun does not reproduce the saved object, so that table says little about normalization.
- The last column of that table barely changes between the two runs. That is expected, not a failure: the dominant genes are still features in the PCA, and they are also the cell-type markers, so the dominance axis stays in the embedding. Only the scaling changed.

## 7. Pearson residuals as a third arm

**How it works.** Analytic Pearson residuals (Lause, Berens and Kobak 2021) replace normalize, log1p and scale. For every gene and cell the model predicts a count:
the gene's total across cells times the cell's total counts, divided by all counts. The residual is `(observed - predicted) / sqrt(predicted + predicted^2 / theta)`,
with `theta = 100` and values clipped at the square root of the number of cells. It is the covariate-free, fixed-parameter version of sctransform.

**What it can't tell.** The prediction still uses the cell's total counts. If the dominant genes take half a cell's counts, the model predicts too much for every other gene,
and their residuals come out negative. Whether that happens in practice is the question for section 8.

This section adds the Pearson arm to the same pipeline as section 6 (PCA, Harmony, neighbours, Leiden) using the same settings and seeds. PCA runs on the residuals directly; no scaling step follows.

In [ ]:
pearson = ad.AnnData(X=raw_counts.copy(), obs=samples_per_cell.copy(), var=pd.DataFrame(index=clustered.var_names))

# Overwrites .X of this in-memory copy with analytic Pearson residuals (theta 100, clipped at sqrt(cells)); .X becomes dense.
sc.experimental.pp.normalize_pearson_residuals(pearson, theta=100)
sc.pp.pca(pearson, n_comps=N_PCS, use_highly_variable=False, random_state=0)

harmony = hm.run_harmony(pearson.obsm['X_pca'].astype('float64'), pearson.obs, 'sample', random_state=0)
harmony_pca = harmony.Z_corr
if harmony_pca.shape[0] != pearson.n_obs:
    harmony_pca = harmony_pca.T
pearson.obsm['X_pca_harmony'] = harmony_pca

sc.pp.neighbors(pearson, use_rep='X_pca_harmony', n_neighbors=15, n_pcs=N_PCS, random_state=0)
sc.tl.leiden(pearson, flavor='igraph', n_iterations=2, directed=False,
             resolution=RESOLUTION, key_added='leiden_rerun', random_state=0)

reruns['pearson residuals']     = pearson
keep_masks['pearson residuals'] = np.ones(pearson.n_obs, dtype=bool)
pearson_clusters = pearson.obs['leiden_rerun'].astype(str).to_numpy()

pd.Series({'cells': pearson.n_obs, 'clusters': len(np.unique(pearson_clusters))}, name='pearson residuals')

**Cluster agreement.** Each row compares two clusterings with the adjusted Rand index. The noise floor row is the control against itself with another Leiden seed.

Because the saved cell-type names are not reliable, "plasma-like cells" here means cells where at least 25% of the counts come from MZB1, TENT5C, SLAMF7, TNFRSF17 and PRDM1.
It is defined from the data, so it does not depend on any name.

In [ ]:
plasma_markers = ['MZB1', 'TENT5C', 'SLAMF7', 'TNFRSF17', 'PRDM1']
marker_columns = [clustered.var_names.get_loc(gene) for gene in plasma_markers]
marker_share   = np.asarray(raw_counts[:, marker_columns].sum(axis=1)).ravel() / size_factors['total counts']
plasma_like    = marker_share >= 0.25

clusterings = {
    'control': control_clusters,
    'control, other seed': control_seed1_clusters,
    'excluding dominant genes': excluding_clusters,
    'pearson residuals': pearson_clusters,
}
pairs = [
    ('control', 'control, other seed'),
    ('control', 'excluding dominant genes'),
    ('control', 'pearson residuals'),
    ('excluding dominant genes', 'pearson residuals'),
]

agreement_rows = []
for first, second in pairs:
    agreement_rows.append({
        'comparison': f'{first} vs {second}',
        'all cells': adjusted_rand_score(clusterings[first], clusterings[second]),
        'plasma-like cells': adjusted_rand_score(clusterings[first][plasma_like], clusterings[second][plasma_like]),
    })

print_table = pd.DataFrame(agreement_rows).set_index('comparison').round(3)
print_table.loc['(plasma-like cells in the data)', 'all cells'] = float(plasma_like.sum())
print_table

## 8. Spike-in test: does the normalization leave the other genes alone?

**The idea.** Real data cannot answer this directly, because cells with a lot of the dominant genes also differ in biology. So the test builds a case where the right answer is known.

Take a random 10% of the cells where the dominant genes are under 10% of the counts. Add counts of the single top gene to each of them, as many as the cell already has, so that gene ends up taking about half the cell.
Nothing else about the cell changes. A good normalization should therefore leave every other gene's value the same before and after.

For each normalization, the notebook takes the mean value of the non-dominant genes in each spiked cell, before and after the spike, and reports the shift in units of that normalization's spread across all cells.
Zero is ideal. A negative shift is the dilution effect.

**What it can't tell.** It is one spike size and one gene, in cells chosen to start with low dominance. The values for the three normalizations are on different scales (log values and residuals),
which is why the shift is divided by the spread.

In [ ]:
rng = np.random.default_rng(0)
low_dominance = np.where(top_fraction_all < 0.10)[0]
perturbed     = rng.choice(low_dominance, size=int(0.10 * len(low_dominance)), replace=False)
spike_gene    = dominant_columns[0]

added_counts  = size_factors['total counts'][perturbed]
spike         = sp.csr_matrix((added_counts, (perturbed, np.full(len(perturbed), spike_gene))), shape=raw_counts.shape)
spiked_counts = (raw_counts + spike).tocsr()

arms   = ['total counts', 'excluding dominant genes', 'pearson residuals']
states = {'before': raw_counts, 'after': spiked_counts}

non_dominant_value = {}
for state, matrix in states.items():
    totals         = np.asarray(matrix.sum(axis=1)).ravel()
    outside_totals = np.asarray(matrix[:, outside_columns].sum(axis=1)).ravel()
    for arm in arms:
        if arm == 'pearson residuals':
            # Builds a temporary in-memory AnnData of this matrix and overwrites its .X with Pearson residuals; nothing is saved.
            residuals = ad.AnnData(X=matrix.copy())
            sc.experimental.pp.normalize_pearson_residuals(residuals, theta=100)
            values = residuals.X
        else:
            size_factor = totals if arm == 'total counts' else outside_totals
            values = (sp.diags(np.median(size_factor) / size_factor) @ matrix).tocsr().log1p()
        non_dominant_value[(arm, state)] = np.asarray(values[:, outside_columns].mean(axis=1)).ravel()

spike_rows = []
shifts     = {}
for arm in arms:
    before = non_dominant_value[(arm, 'before')]
    after  = non_dominant_value[(arm, 'after')]
    shifts[arm] = (after[perturbed] - before[perturbed]) / before.std()
    spike_rows.append({
        'normalization': arm,
        'spiked cells': len(perturbed),
        'mean shift in SDs': shifts[arm].mean(),
        'share of spiked cells shifted down > 0.5 SD': (shifts[arm] < -0.5).mean(),
    })

pd.DataFrame(spike_rows).set_index('normalization').round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, arm in zip(axes, arms):
    ax.hist(shifts[arm], bins=60, color='steelblue')
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('shift in mean non-dominant value (SDs)')
    ax.set_title(arm)

plt.show()

**The same check on the real data.** For every cell, the mean value of the non-dominant genes under each normalization, in bins of the dominant-gene fraction (lines show the bin medians, as z-scores).
A line that falls as the dominant fraction rises is what dilution looks like. Part of any slope here is real biology, since cells with a high dominant fraction are also different cell types,
so compare the lines with each other, not with a flat line.

In [ ]:
bin_edges = np.linspace(0, 1, 11)
bin_index = np.digitize(top_fraction_all, bin_edges[1:-1])

fig, ax = plt.subplots(figsize=(7, 4.5))
for arm in arms:
    values = non_dominant_value[(arm, 'before')]
    z      = (values - values.mean()) / values.std()
    medians = [np.median(z[bin_index == b]) if (bin_index == b).sum() > 100 else np.nan for b in range(10)]
    ax.plot((bin_edges[:-1] + bin_edges[1:]) / 2, medians, marker='o', label=arm)

ax.set_xlabel('fraction of counts from the dominant genes')
ax.set_ylabel('median non-dominant value (z-score)')
ax.set_title('Non-dominant genes against dominance')
ax.legend()
plt.show()

**Reading the result.**

- In the spike-in table, the shift for *total counts* is the dilution effect. *Excluding dominant genes* is zero by construction, because its size factor does not include the spiked gene. It is the reference for what a correction does.
  If *pearson residuals* is close to *total counts*, the residuals have the same dilution problem. If it is close to zero, the residuals avoid it.
- In the agreement table, compare *control vs pearson residuals* with the noise floor row (control vs control, other seed). A value near or above the floor means the clusters are not much affected by switching to residuals.
- The plasma-like subset is defined by the dominant markers, so it picks out cells with high dominance by construction.